In [40]:
# Parameters
megadescriptor_version = 'T-224'  # 'S-224', 'B-224', 'L-384'
detection = '' # '', _detected', '_detected_manual'
seed = 42
query_ratio = 0.2

In [41]:
import torch
import numpy as np
import joblib
import torch.nn as nn
import torch.optim as optim
import random
from collections import defaultdict
import sys
sys.path.append(r'C:\BP\pythonProject1')
from misclassification_utils import show_misclassified

In [42]:
np.random.seed(seed)

# Path to new file
data_path = f"saved_models/{megadescriptor_version}/data{detection}.npz"
encoder_path = f"saved_models/{megadescriptor_version}/label_encoder{detection}.pkl"

# Load everything at once
data = np.load(data_path)

embeddings = data["embeddings"]      # shape (N, D)
labels = data["label_ids"]           # integer labels
# original_labels = data["labels"]     # string labels (optional)

print("Embeddings shape:", embeddings.shape)

# Optional: load encoder if you want inverse_transform
encoder = joblib.load(encoder_path)
names = encoder.inverse_transform(labels)

encoder = joblib.load(encoder_path)
id_to_name = dict(enumerate(encoder.classes_))
name_to_id = {v: k for k, v in id_to_name.items()}


Embeddings shape: (319, 768)


In [43]:
from sklearn.model_selection import train_test_split

# Create an array of original indices to track which embedding each sample came from
original_indices = np.arange(len(embeddings))

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    embeddings, labels, original_indices, test_size=query_ratio, random_state=seed
)
import torch
from torch.utils.data import TensorDataset, DataLoader

# convert numpy arrays from the train/test split into torch tensors
# use float32 for the embeddings and long for the integer labels
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [44]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# convert numpy arrays from the train/test split into torch tensors
# use float32 for the embeddings and long for the integer labels
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

# optional: keep full dataset tensors for evaluation later
X = torch.tensor(embeddings, dtype=torch.float32)
y = torch.tensor(labels, dtype=torch.long)

# build training dataset and loader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)


In [45]:
class Classifier(nn.Module):
    def __init__(self, input_dim=768, num_classes=10, hidden_dim=256, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x):
        return self.net(x)


In [46]:
# determine number of classes from training labels
num_classes = len(torch.unique(torch.tensor(y_train)))

model = Classifier(input_dim=768, num_classes=num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(100):
    model.train()
    total_loss = 0
    correct = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (outputs.argmax(1) == y_batch).sum().item()

    acc = correct / len(train_dataset)
    print(f"Epoch {epoch}: loss={total_loss:.3f}, acc={acc:.3f}")


Epoch 0: loss=39.847, acc=0.220
Epoch 1: loss=33.112, acc=0.365
Epoch 2: loss=27.892, acc=0.510
Epoch 3: loss=22.541, acc=0.580
Epoch 4: loss=18.550, acc=0.655
Epoch 5: loss=14.222, acc=0.761
Epoch 6: loss=11.256, acc=0.788
Epoch 7: loss=9.434, acc=0.843
Epoch 8: loss=6.675, acc=0.902
Epoch 9: loss=5.393, acc=0.910
Epoch 10: loss=4.208, acc=0.922
Epoch 11: loss=3.744, acc=0.933
Epoch 12: loss=2.266, acc=0.961
Epoch 13: loss=1.998, acc=0.973
Epoch 14: loss=2.438, acc=0.969
Epoch 15: loss=1.951, acc=0.961
Epoch 16: loss=2.081, acc=0.953
Epoch 17: loss=1.539, acc=0.984
Epoch 18: loss=1.292, acc=0.988
Epoch 19: loss=0.717, acc=1.000
Epoch 20: loss=0.546, acc=1.000
Epoch 21: loss=0.568, acc=1.000
Epoch 22: loss=0.576, acc=0.996
Epoch 23: loss=0.878, acc=0.980
Epoch 24: loss=0.543, acc=0.996
Epoch 25: loss=0.410, acc=1.000
Epoch 26: loss=0.252, acc=1.000
Epoch 27: loss=0.287, acc=0.996
Epoch 28: loss=0.288, acc=0.996
Epoch 29: loss=0.218, acc=1.000
Epoch 30: loss=0.221, acc=1.000
Epoch 31: l

In [47]:
# Evaluate on training set
model.eval()
with torch.no_grad():
    preds = model(X_train_tensor).argmax(1)
    accuracy = (preds == y_train_tensor).float().mean()
    print("Final train accuracy:", accuracy.item())

    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    print("Final train loss:", loss.item())

Final train accuracy: 1.0
Final train loss: 4.57100904895924e-05


In [48]:
# Check misclassified samples on training set
show_misclassified(y_train_tensor, preds, idx_train, detection, encoder)

Number wrong: 0


## CrossEntropy Loss


In [51]:
# Evaluate on validation set
model.eval()
with torch.no_grad():
    preds_test = model(X_test_tensor).argmax(1)
    accuracy_test = (preds_test == y_test_tensor).float().mean()
    print("Final validation accuracy:", accuracy_test.item())

    outputs_test = model(X_test_tensor)
    loss_test = criterion(outputs_test, y_test_tensor)
    print("Final validation loss:", loss_test.item())

Final validation accuracy: 0.640625
Final validation loss: 2.183112144470215


In [50]:
# reuse helper function defined earlier to list misclassified samples on validation set
show_misclassified(y_test_tensor, preds_test, idx_test, detection, encoder)

Number wrong: 23
Index 6 (Orig 176): rys_trening_data_Beno\Izidor\Izidor_18.JPG
  True: Izidor, Predicted: Edo

Index 7 (Orig 186): rys_trening_data_Beno\Izidor\Izidor_27.JPG
  True: Izidor, Predicted: Benadik

Index 12 (Orig 164): rys_trening_data_Beno\Eliska\Eliska_7.JPG
  True: Eliska, Predicted: Adam

Index 14 (Orig 211): rys_trening_data_Beno\Kiara\Kiara_17.JPG
  True: Kiara, Predicted: Roman

Index 16 (Orig 185): rys_trening_data_Beno\Izidor\Izidor_26.JPG
  True: Izidor, Predicted: Roman

Index 18 (Orig 314): rys_trening_data_Beno\Zora\Zora_5.JPG
  True: Zora, Predicted: Milos

Index 21 (Orig 177): rys_trening_data_Beno\Izidor\Izidor_19.JPG
  True: Izidor, Predicted: Zora

Index 22 (Orig 197): rys_trening_data_Beno\Izidor\Izidor_37.JPG
  True: Izidor, Predicted: Edo

Index 23 (Orig 108): rys_trening_data_Beno\Brano\Brano_2.JPG
  True: Brano, Predicted: Edo

Index 25 (Orig 118): rys_trening_data_Beno\Dio\Dio_13.jpg
  True: Dio, Predicted: Lubos

Index 26 (Orig 296): rys_trening_da

## Poznamenanie k výsledkom tréningu

- **Izidor_27** (nočná fotka zozadu) bol nesprávne klasifikovaný ako **Miloš**, ktorý má v datasete veľa obrázkov zozadu, ale aj veľa nočných.
- **Eliška_7** sa pravdepodobne podobá na **Braňa**.
- **Kiara_17** je nočný dobre osvetlený záber zboku s kontrastným zatmeným pozadím, veľmi podobný mnohým zaberom **Romana** s týmito charakteristikami.
- **Izidor_26** je záber zboku s výnimočne zeleným pozadím, nesprávne klasifikovaný ako **Roman**, ktorý má v datasete (v porovnaní s ostatnými) výrazne veľa snímok zboku.
- **Zora_5** bola pre kombináciu sneh + ihličnany klasifikovaná ako **Izidor**, ktorý má v tréningovom sete veľa obrázkov tohto typu.
- **Izidor_37** (nočná fotka + svietiace oči) bol klasifikovaný ako **Miloš**, ktorý má v datasete veľa obrázkov s touto kombináciou.
- **Brano_2** (jesenná fotka) bol nesprávne klasifikovaný ako **Eliška**, u ktorej sú niektoré jesenné obrázky.